# Z Fighting in Three JS

If you like computer graphics, this is the project for you

-----

## The Software

Three.js is a JavaScript library used to render 3D graphics on web browsers using WebGL and WebGPU. It lets the developer use various common 3D tool like lights, materials, meshes, and shader scripts without exposing all the underlying rendering logic. Its audience is web developers, artists, game developers, data visualization, mapping, and education. My project will focus on Z-fighting, which, as seen in the example below, happens on different scales.

Why does Z-Fighting matter? There are multiple use cases where developers have large scales on their project where worlds can be huge and rendering issues begin to occur. Think: VR Experiences, Interactive Maps, Open-Worlds. Choices then are made of what methods are best appropriate while trading off speed and accuracy.

## The Problem

When rendering a 3D scene, Three.js creates a virtual camera that defines the space it will render in terms of two variables near (n) and far (f). These two variables determine how far, in meters, the camera will process. However, meters means nothing to the graphics engine (it's arbitrary), and instead, it's much more helpful for it to normalize the two distances through a formula that takes f and n and returns a value between 0 and 1. This gives the rendering engine a proportion for distance, which is much more concrete and useful for whatever wibbly wobbly, timey wimey stuff it does later on.

Sounds simple, right? Well, it is until you come to the decision of how to implement the function. Three.js has three (hah) separate functions that create this proportion. The default, the reversed, and the logarithmic. To complicate things further, Three.js also has to decide just how to round the distance values returned by the inverse function i.e how to turn the proportion 0.876431, into a legible distance in meters.

These two decisions combined, affect how objects farther and farther away from the camera behave, and in a scene with hundreds of thousands of objects, how expensive the calculations might become.

Does this sound familiar? It should because it's a classic problem of conditioning, which is what I would like to explore in this project: Comparing and contrasting the three functions and the rounding to determine the tolerances and use cases of each.

Here is a video from the Three.js repository where they exemplify this very same problem.

![Red stripes appear as the planes move](img/Z-Fight.gif)

## The Math: Distances

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# The Camera Settings -> How far and close the camera will render a scene (imagine you're placing two separate pieces of paper in front of the camera at separate distances, the area in between is what the camera will "see")
near, far = 0.1, 1000.0

# The byte buffer size
B = 2**24 - 1
# The unit roundoff
u = 2.0**-24

The camera will turn a number $d$ into a number $w$ between $[0,1]$. Three.js lets the developer choose between the methods below, and each has an inverse function that will take a number between $[0,1]$ and return a distance.

Standard formula: $\dfrac{f(d-n)}{(f-n)d}$

Logarithmic formula: $\dfrac{\log_2(1+d)}{\log_2(1+f)}$

Reversed formula: $\dfrac{n(f-d)}{(f-n)d}$

Written out in code:

In [2]:
#Standard
def convert_distance_to_range_standard(distance): return far * (distance - near) / ((far - near) * distance)
def convert_range_to_distance_standard(normal_d): return far * near / (far - normal_d * (far - near))

#Logarithmic
def convert_distance_to_range_logarithmic(distance): return np.log2(1 + distance) / np.log2(1 + far)
def convert_range_to_distance_logarithmic(normal_d): return (1 + far) ** normal_d - 1

#Reversed
def convert_distance_to_range_reversed(distance): return near * (far - distance) / ((far - near) * distance)
def convert_range_to_distance_reversed(normal_d): return near * far / (near + normal_d * (far - near))

## The Math: Rounding

Since the value $w$ in range $[0,1]$ might be representing a value of 7 orders of magnitude, it can't be stored precisely, so it is rounded to the nearest by the formula $k/B$ where B is the 24-bit integer buffer and stored in evenly spaced steps.

The smallest gap between two distances the buffer will be able to see is the change in these two steps, and so two surfaces really close to each other will sit in the same bucket and Z-Fighting will ensue – To the program, the two surfaces are sitting in the same spot.

In [ ]:
# WIP: Representing the gaps in code for float32 and integer

## The Math: Condition Number

We know from lecture that condition number for $d(w)$ will be

$\kappa =\frac{w\,d'(w)}{d(w)}$

In [3]:
# WIP Derivatives and example

## Trade-offs

WIP: How exactly do the different functions scale in cost?

## Question and experiment

### Logarithmic function doesn't use the near variable

Does using different near values affect precision when using the logarithmic setting, does it make sense that it would?

### Measuring accuracy and cost for each depth setting on our respective laptops and browsers

X-Axis: What is the smallest gap that stays solid at 100 m, 0.1m etc...

Y-Axis: Different fps with varying hidden layers